In [1]:
from pathlib import Path
from zipfile import ZipFile
import io
import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path("/home/gdanzil/econ470/a0/work/hcris-data")

In [3]:
HCRIS_VARS = [
    # (name, wksht_cd, line_num, clmn_num, source)
    ("beds",                       "S300001", "01400", "00200", "numeric"),
    ("tot_charges",                "G300000", "00100", "00100", "numeric"),
    ("tot_discounts",              "G300000", "00200", "00100", "numeric"),
    ("net_pat_rev",                "G300000", "00300", "00100", "numeric"),
    ("tot_operating_exp",          "G300000", "00400", "00100", "numeric"),
    ("ip_charges",                 "G200000", "00100", "00100", "numeric"),
    ("icu_charges",                "G200000", "01600", "00100", "numeric"),
    ("ancillary_charges",          "G200000", "01800", "00100", "numeric"),
    ("tot_discharges",             "S300001", "00100", "01500", "numeric"),
    ("mcare_discharges",           "S300001", "00100", "01300", "numeric"),
    ("mcaid_discharges",           "S300001", "00100", "01400", "numeric"),
    ("tot_mcare_payment",          "E00A18A", "05900", "00100", "numeric"),
    ("secondary_mcare_payment",    "E00A18A", "06000", "00100", "numeric"),
    ("street",                     "S200001", "00100", "00100", "alpha"),
    ("city",                       "S200001", "00200", "00100", "alpha"),
    ("state",                      "S200001", "00200", "00200", "alpha"),
    ("zip",                        "S200001", "00200", "00300", "alpha"),
    ("county",                     "S200001", "00200", "00400", "alpha"),
    ("name",                       "S200001", "00300", "00100", "alpha"),
    ("hvbp_payment",               "E00A18A", "07093", "00100", "numeric"),
    ("hrrp_payment",               "E00A18A", "07094", "00100", "numeric"),
    ("tot_uncomp_care_charges",    "S100000", "02000", "00300", "numeric"),
    ("tot_uncomp_care_partial_pmts","S100000", "02200", "00300", "numeric"),
    ("bad_debt",                   "S100000", "02800", "00100", "numeric"),
    ("cost_to_charge",             "S100000", "00100", "00100", "numeric"),
    ("new_cap_ass",                "A700001", "01000", "00200", "numeric"),
    ("cash",                       "G000000", "00100", "00100", "numeric"),
    ("fixed_assets",               "G000000", "03000", "00100", "numeric"),
    ("depr_land",                  "G000000", "01400", "00100", "numeric"),
    ("depr_bldg",                  "G000000", "01600", "00100", "numeric"),
    ("depr_lease",                 "G000000", "01800", "00100", "numeric"),
    ("depr_fixed_equip",           "G000000", "02000", "00100", "numeric"),
    ("depr_auto",                  "G000000", "02200", "00100", "numeric"),
    ("depr_major_equip",           "G000000", "02400", "00100", "numeric"),
    ("depr_minor_equip",           "G000000", "02600", "00100", "numeric"),
    ("depr_HIT",                   "G000000", "02800", "00100", "numeric"),
    ("current_assets",             "G000000", "01100", "00100", "numeric"),
    ("current_liabilities",        "G000000", "04500", "00100", "numeric"),
]

In [4]:
RPT_COLS = [
    "RPT_REC_NUM", "PRVDR_CTRL_TYPE_CD", "PRVDR_NUM", "NPI",
    "RPT_STUS_CD", "FY_BGN_DT", "FY_END_DT", "PROC_DT",
    "INITL_RPT_SW", "LAST_RPT_SW", "TRNSMTL_NUM", "FI_NUM",
    "ADR_VNDR_CD", "FI_CREAT_DT", "UTIL_CD", "NPR_DT",
    "SPEC_IND", "FI_RCPT_DT",
]

In [5]:
LONG_COLS = ["RPT_REC_NUM", "WKSHT_CD", "LINE_NUM", "CLMN_NUM", "ITM_VAL_NUM"]

In [6]:
def _resolve_path(year, suffix):
    """Find the CSV file for a given year and suffix (alpha/nmrc/rpt).

    Tries unzipped folders first (two naming conventions), then ZIP archives.
    Returns (path_or_buffer, label) where label is for logging.
    """
    base = PROJECT_ROOT / "data" / "input" / "HCRIS_v2010"

    folders = [f"HospitalFY{year}", f"HOSP10FY{year}"]
    files = [
        f"hosp10_{year}_{suffix.upper()}.CSV",
        f"HOSP10_{year}_{suffix.lower()}.csv",
    ]
    for folder in folders:
        for fname in files:
            path = base / folder / fname
            if path.exists():
                return path, str(path)

    # ZIP fallback
    zips = [
        f"HOSP10FY{year}.ZIP", f"HOSP10FY{year}.zip",
        f"HospitalFY{year}.ZIP", f"HospitalFY{year}.zip",
    ]
    entries = [
        f"HOSP10_{year}_{suffix.lower()}.csv",
        f"hosp10_{year}_{suffix.upper()}.CSV",
    ]
    for zf in zips:
        zip_path = base / zf
        if zip_path.exists():
            with ZipFile(zip_path) as zfile:
                names = zfile.namelist()
                for entry in entries:
                    if entry in names:
                        buf = io.BytesIO(zfile.read(entry))
                        return buf, f"{zip_path}:{entry}"

    raise FileNotFoundError(f"Cannot find HCRIS v2010 data for year {year}, suffix {suffix}")


def _read_csv(year, suffix, col_names):
    source, label = _resolve_path(year, suffix)
    return pd.read_csv(source, header=None, names=col_names, dtype=str)

In [8]:
def _resolve_path(year, suffix):
    base = PROJECT_ROOT / "HCRIS_v2010"

    folders = [f"HospitalFY{year}", f"HOSP10FY{year}"]
    files = [
        f"hosp10_{year}_{suffix.upper()}.CSV",
        f"HOSP10_{year}_{suffix.lower()}.csv",
    ]

    for folder in folders:
        for fname in files:
            path = base / folder / fname
            if path.exists():
                return path, str(path)

    zips = [
        f"HOSP10FY{year}.ZIP", f"HOSP10FY{year}.zip",
        f"HospitalFY{year}.ZIP", f"HospitalFY{year}.zip",
    ]
    entries = [
        f"HOSP10_{year}_{suffix.lower()}.csv",
        f"hosp10_{year}_{suffix.upper()}.CSV",
    ]

    for zf in zips:
        zip_path = base / zf
        if zip_path.exists():
            with ZipFile(zip_path) as zfile:
                names = zfile.namelist()
                for entry in entries:
                    if entry in names:
                        buf = io.BytesIO(zfile.read(entry))
                        return buf, f"{zip_path}:{entry}"

    raise FileNotFoundError(f"Cannot find HCRIS v2010 data for year {year}, suffix {suffix}")

def _read_csv(year, suffix, col_names):
    source, label = _resolve_path(year, suffix)
    print("Reading:", label)
    return pd.read_csv(source, header=None, names=col_names, dtype=str)

In [9]:
alpha2010 = _read_csv(2010, "alpha", LONG_COLS)
alpha2010.head()

Reading: /home/gdanzil/econ470/a0/work/hcris-data/HCRIS_v2010/HospitalFY2010/hosp10_2010_ALPHA.CSV


,RPT_REC_NUM,WKSHT_CD,LINE_NUM,CLMN_NUM,ITM_VAL_NUM
0,235,A000000,00100,00000,00100CAP REL COSTS-BLDG & FIXT
1,235,A000000,00200,00000,00200CAP REL COSTS-MVBLE EQUIP
2,235,A000000,00300,00000,00300OTHER CAP REL COSTS
3,235,A000000,00400,00000,00400EMPLOYEE BENEFITS
4,235,A000000,00500,00000,00500ADMINISTRATIVE & GENERAL


In [11]:
def _resolve_path(year, suffix):
    base = PROJECT_ROOT / "HCRIS_v2010"

    folders = [f"HospitalFY{year}", f"HOSP10FY{year}"]
    files = [
        f"hosp10_{year}_{suffix.upper()}.CSV",
        f"HOSP10_{year}_{suffix.lower()}.csv",
    ]

    for folder in folders:
        for fname in files:
            path = base / folder / fname
            if path.exists():
                return path, str(path)

    zips = [
        f"HOSP10FY{year}.ZIP", f"HOSP10FY{year}.zip",
        f"HospitalFY{year}.ZIP", f"HospitalFY{year}.zip",
    ]
    entries = [
        f"HOSP10_{year}_{suffix.lower()}.csv",
        f"hosp10_{year}_{suffix.upper()}.CSV",
    ]

    for zf in zips:
        zip_path = base / zf
        if zip_path.exists():
            with ZipFile(zip_path) as zfile:
                names = zfile.namelist()
                for entry in entries:
                    if entry in names:
                        buf = io.BytesIO(zfile.read(entry))
                        return buf, f"{zip_path}:{entry}"

    raise FileNotFoundError(f"Cannot find HCRIS v2010 data for year {year}, suffix {suffix}")


def _read_csv(year, suffix, col_names):
    source, label = _resolve_path(year, suffix)
    print("Reading:", label)
    return pd.read_csv(source, header=None, names=col_names, dtype=str)


def extract_v2010():
    print("Extracting HCRIS v2010 (2010-2019)...")
    frames = []

    for yr in range(2010, 2020):
        alpha = _read_csv(yr, "alpha", LONG_COLS)
        nmrc = _read_csv(yr, "nmrc", LONG_COLS)
        rpt = _read_csv(yr, "rpt", RPT_COLS)

        for df in [alpha, nmrc]:
            for c in ["RPT_REC_NUM", "WKSHT_CD", "LINE_NUM", "CLMN_NUM", "ITM_VAL_NUM"]:
                df[c] = df[c].astype(str).str.strip().str.upper()

        for c in ["RPT_REC_NUM", "PRVDR_NUM", "NPI", "FY_BGN_DT", "FY_END_DT",
                  "PROC_DT", "FI_CREAT_DT", "RPT_STUS_CD"]:
            rpt[c] = rpt[c].astype(str).str.strip()

        final = rpt[[
            "RPT_REC_NUM", "PRVDR_NUM", "NPI", "FY_BGN_DT", "FY_END_DT",
            "PROC_DT", "FI_CREAT_DT", "RPT_STUS_CD"
        ]].copy()

        final.columns = [
            "report", "provider_number", "npi", "fy_start", "fy_end",
            "date_processed", "date_created", "status"
        ]
        final["year"] = yr
        final["data_source"] = "v2010"

        for var_name, wksht, line, col, source in HCRIS_VARS:
            data = alpha if source == "alpha" else nmrc

            vals = (
                data.loc[
                    (data["WKSHT_CD"] == wksht) &
                    (data["LINE_NUM"] == line) &
                    (data["CLMN_NUM"] == col),
                    ["RPT_REC_NUM", "ITM_VAL_NUM"]
                ]
                .rename(columns={"RPT_REC_NUM": "report", "ITM_VAL_NUM": var_name})
                .drop_duplicates(subset=["report"])
            )

            if source == "numeric":
                vals[var_name] = pd.to_numeric(vals[var_name], errors="coerce")

            final = final.merge(vals, on="report", how="left")

        frames.append(final)
        print(f"  v2010 {yr}: {len(final):,} reports")

    combined = pd.concat(frames, ignore_index=True)

    outdir = Path.cwd() / "output"
    outdir.mkdir(parents=True, exist_ok=True)
    combined.to_csv(outdir / "HCRIS_Data_v2010.txt", sep="\t", index=False)

    print(f"Saved to: {outdir / 'HCRIS_Data_v2010.txt'}")
    print(f"Rows: {len(combined):,}")
    print(f"Cols: {combined.shape[1]}")
    return combined

In [12]:
v2010 = extract_v2010()

Extracting HCRIS v2010 (2010-2019)...
Reading: /home/gdanzil/econ470/a0/work/hcris-data/HCRIS_v2010/HospitalFY2010/hosp10_2010_ALPHA.CSV
Reading: /home/gdanzil/econ470/a0/work/hcris-data/HCRIS_v2010/HospitalFY2010/hosp10_2010_NMRC.CSV
Reading: /home/gdanzil/econ470/a0/work/hcris-data/HCRIS_v2010/HospitalFY2010/hosp10_2010_RPT.CSV
  v2010 2010: 2,322 reports
Reading: /home/gdanzil/econ470/a0/work/hcris-data/HCRIS_v2010/HospitalFY2011/hosp10_2011_ALPHA.CSV
Reading: /home/gdanzil/econ470/a0/work/hcris-data/HCRIS_v2010/HospitalFY2011/hosp10_2011_NMRC.CSV
Reading: /home/gdanzil/econ470/a0/work/hcris-data/HCRIS_v2010/HospitalFY2011/hosp10_2011_RPT.CSV
  v2010 2011: 6,143 reports
Reading: /home/gdanzil/econ470/a0/work/hcris-data/HCRIS_v2010/HospitalFY2012/hosp10_2012_ALPHA.CSV
Reading: /home/gdanzil/econ470/a0/work/hcris-data/HCRIS_v2010/HospitalFY2012/hosp10_2012_NMRC.CSV
Reading: /home/gdanzil/econ470/a0/work/hcris-data/HCRIS_v2010/HospitalFY2012/hosp10_2012_RPT.CSV
  v2010 2012: 6,202 repo

In [13]:
wanted = [
    "provider_number", "fy_start", "fy_end", "beds", "tot_charges",
    "tot_mcare_payment", "hrrp_payment", "hvbp_payment"
]
print("Missing:", [c for c in wanted if c not in v2010.columns])
v2010[[c for c in wanted if c in v2010.columns]].head()

Missing: []


,provider_number,fy_start,fy_end,beds,tot_charges,tot_mcare_payment,hrrp_payment,hvbp_payment
0,492007,06/01/2010,05/31/2011,35.0,28099723.0,NaN,NaN,NaN
1,044017,07/01/2010,06/30/2011,16.0,22570813.0,NaN,NaN,NaN
2,193070,07/01/2010,06/30/2011,15.0,12710268.0,NaN,NaN,NaN
3,191311,07/01/2010,06/30/2011,25.0,24562552.0,NaN,NaN,NaN
4,341311,07/01/2010,06/30/2011,25.0,37222309.0,NaN,NaN,NaN
